# Test Functionality of GestionOt{class}

### Pasos para calificar una actividad.


1. Separar los eventos que tienen alimentador de los que no.
   
   1.1. Separar y calificar aquellos que son de TRANSPORTE, ALIMENTACIÓN, SE LABORA, INFO, se repite en la calificación
   
2. A los eventos que si tienen alimentador.
   
   2.1. Separar aquellos que sabemos que son SAPG, los más fáciles de identificar.

   2.2. Separar aquellos que son de Servicios Ocasionales.

   2.3. Calificar usando la Red Neuronal.

In [1]:

from eerssa import gestionOT
from eerssa import matrizActividades
from pprint import pprint
from pathlib import Path
import pandas as pd

test_path = '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/'
list_pdfs = []
for path in Path( test_path ).glob("**/*.pdf"):
  list_pdfs.append( str(path) )
  list_pdfs.sort()


Success!!!


### Secuencial GLOBAL LOCK

In [ ]:
### Secuencial en un solo procesador.
obj_lists = []
for file in list_pdfs:
  ot = gestionOT.GestionOt( file )
  ot.load_ot()
  obj_lists.append( ot )

### DASK ?

#### Primer intento

Primero creo un cluster locar de computación

In [15]:
from dask.distributed import LocalCluster
client = LocalCluster().get_client()

/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 42977 instead
  warnings.warn(


Luego, con el listado de OT's de ejecutado en primera instancia, genero objetos en los cluster de computación local, 

Ejecuto la función `load_ot()` y guardo los resultados a la misma lista de objetos

In [16]:

futures = [client.submit(gestionOT.GestionOt, file, actor=True) for file in list_pdfs ]
ot_array = [future.result() for future in futures]

ot_cargada = [ot.load_ot() for ot in ot_array]
obj_lists = [future.result() for future in ot_cargada]

Success!!!
Success!!!
Success!!!
Success!!!


In [17]:
rta[3].data

{'version': '0.12.0',
 'link': '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/12_NL_Tres_hojas_ultima_vacia.pdf',
 'id_ot': 134508,
 'exito': True,
 'cuadrilla': 'Yacuambi Z1 (Cuadrilla. Nro. 8)',
 'responsable': ['LOZANO SIGCHO NAUN ENRIQUE', 'JECE'],
 'colaboradores': {'total': 3,
  'nombres': [['CABRERA GONZALEZ LUIS RENE', 'LNI1'],
   ['ERAS POMA SANTOS FERNANDO', 'LNI1'],
   ['PUCHAICELA CASTILLO JORGE ALEXANDER', 'LIN1']]},
 'diaSemana': 'lunes',
 'fecha': datetime.datetime(2024, 7, 29, 0, 0, tzinfo=<DstTzInfo 'America/Guayaquil' -05-1 day, 19:00:00 STD>),
 'fechaInicio': 'lunes, 29 de julio del 2024',
 'fechaFinal': '29/07/2024 20:40:00',
 'sitio': 'Yacuambi - Tamboloma, Hucapamba y Jembuentza',
 'descripcion': 'Traslado a Tamboloma para revisar sector sin servicio eléctrico derivación a La Florida luego a Huacapamba\nrevisar derivación a Playas sin servicio eléctrico. Atender daños por llamado de Centro de  Control',
 'tEstimado': '8',
 'vehiculo': {'numero': 'R-171

In [20]:
nro_ot = 3
obj_lists[nro_ot].data['link']

'/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/12_NL_Tres_hojas_ultima_vacia.pdf'

In [21]:
obj_lists[nro_ot].data['actividades']

[{'Item': '1',
  'Actividad': 'PROG',
  'Evento': 'En Bodega Yacuambi se coordina trabajos con los compañeros,\nlegalización de OT, se revisa herramientas y materiales,',
  'Ali': None,
  'Alimentador': None,
  'Tipo': 'RUTINARIA',
  'InicioEvento': '2024-07-29 08:00:00',
  'FinEvento': '2024-07-29 08:30:00'},
 {'Item': '2',
  'Actividad': 'NO PROG',
  'Evento': 'En Yacuambi sector del Estadio Municipal se retira acometida\nprovicionalmente del Med 1000376278 a pedido del cliente por\ncontruccion de nueva vivienda',
  'Ali': 'ALI',
  'Alimentador': 'Yacuambi',
  'Tipo': 'PREDICTIVO',
  'InicioEvento': '2024-07-29 08:30:00',
  'FinEvento': '2024-07-29 09:10:00'},
 {'Item': '3',
  'Actividad': 'TRANSP',
  'Evento': 'DAÑO REPORTADO POR CC. Mensaje de CC. Traslado desde Yacuambi hasta\nTamboloma informan derivacion a La Florida sin servicio eléctrico',
  'Ali': None,
  'Alimentador': None,
  'Tipo': 'TRANSPORTE',
  'InicioEvento': '2024-07-29 09:10:00',
  'FinEvento': '2024-07-29 10:00:00'

In [22]:
matriz_test = matrizActividades.ConvertirOT_a_ActividadesCSV(  obj_lists[nro_ot] )
matriz_test[['Cuenta','Evento','Tipo','Actividad','Alimentador','Fecha']]

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [5]:
matriz_test[['Cuenta','Evento']]

,Cuenta,Evento
0,Red_aerea,En la agencia de la EERSSA El Pangui se coordi...
1,Red_aerea,"Pangui en el centro de la ciudad, calle Quito ..."
3,Alumbrado,RECLAMO No. 1100629005 03-07-24/12:05. Pangui ...
6,Alumbrado,RECLAMO No. 1100629005 03-07-24/12:05. Pangui ...
9,transporte,Nos trasladamos desde el Pangui hacia Pachicutza.
10,Alumbrado,En el sector de Pachicutza junto al parterre d...
11,transporte,Nos trasladamos desde el sector de Pachicutza ...
12,Alumbrado,"En el sector El Padmi, en la estructura. No. 9..."
13,Alumbrado,En el sector de Los Encuentros estructura. No....
14,lunch,Lunch el sector de Los Encuentros.


## Probar DASK

## Test dierctory

In [3]:
from dask.distributed import LocalCluster
client = LocalCluster().get_client()


In [7]:
## ¿Como se usa DASK para creacion de objetos?

ots_raw = []
for file in list_pdfs:
  this_ot = client.submit( gestionOT.GestionOt, file )
  conversion = client.submit( gestionOT.GestionOt, file )

In [3]:
obj_lists = []
obj_data  = []
for file in list_pdfs:
  ot = gestionOT.GestionOt( file )
  ot.load_ot()
  obj_lists.append( ot )
  obj_data.append( ot.data )

In [5]:
len(obj_data)

29

In [4]:
obj_lists[3].log

[{'t': '2025-01-17T03:49:16.915439',
  'level': 'INFO',
  'message': 'Creacion de la OT',
  'detail': 'Ninguno'}]

In [4]:
ot_as_pd = pd.DataFrame( obj_data )

In [15]:
dbg

version                                                     0.12.0
link             /home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/te...
id_ot                                                     134508.0
exito                                                         True
cuadrilla                          Yacuambi Z1 (Cuadrilla. Nro. 8)
responsable                     [LOZANO SIGCHO NAUN ENRIQUE, JECE]
colaboradores    {'total': 3, 'nombres': [['CABRERA GONZALEZ LU...
diaSemana                                                    lunes
fecha                                    2024-07-29 00:00:00-05:00
fechaInicio                            lunes, 29 de julio del 2024
fechaFinal                                     29/07/2024 20:40:00
sitio                 Yacuambi - Tamboloma, Hucapamba y Jembuentza
descripcion      Traslado a Tamboloma para revisar sector sin s...
tEstimado                                                        8
vehiculo         {'numero': 'R-171', 'placa': 'AAA-4278', 'mar